In [ ]:
!python /content/drive/MyDrive/AUV_Data/Data_gen_single_object.py

In [ ]:
import cv2
import numpy as np
import random
import os

# ==================================================
# BASE PATH
# ==================================================
BASE_DIR = "/content/drive/MyDrive/AUV_Data/underwater_single_object"

for split in ["train", "val"]:
    os.makedirs(os.path.join(BASE_DIR, f"images/{split}"), exist_ok=True)
    os.makedirs(os.path.join(BASE_DIR, f"labels/{split}"), exist_ok=True)

# ==================================================
# BACKGROUND
# ==================================================
def underwater_bg(w=640, h=640):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:] = (
        random.randint(80, 140),
        random.randint(100, 170),
        random.randint(30, 80)
    )
    return cv2.GaussianBlur(img, (15, 15), 0)

# ==================================================
# GATE (class 0)
# ==================================================
def draw_gate(img):
    h, w, _ = img.shape
    size = random.randint(240, 360)
    thickness = random.randint(8, 16)

    cx, cy = random.randint(200, 440), random.randint(200, 440)
    angle = random.uniform(-20, 20)

    rect = ((cx, cy), (size, size), angle)
    box = cv2.boxPoints(rect).astype(int)

    cv2.drawContours(img, [box], 0, (0, 165, 255), thickness)
    x, y, bw, bh = cv2.boundingRect(box)

    return img, [0, (x+bw/2)/w, (y+bh/2)/h, bw/w, bh/h]

# ==================================================
# MAT (class 1)
# ==================================================
def draw_mat(img):
    h, w, _ = img.shape
    mw, mh = random.randint(140, 220), random.randint(100, 180)
    x, y = random.randint(200, 420), random.randint(400, 560)

    cv2.rectangle(img, (x, y), (x+mw, y+mh), (40, 40, 40), -1)

    return img, [1, (x+mw/2)/w, (y+mh/2)/h, mw/w, mh/h]

# ==================================================
# DRUM (class 2)
# ==================================================
def draw_drum(img):
    h, w, _ = img.shape
    r = random.randint(35, 65)
    cx, cy = random.randint(120, 520), random.randint(160, 480)

    color = random.choice([(0, 0, 255), (255, 0, 0)])
    cv2.circle(img, (cx, cy), r, color, -1)

    return img, [2, cx/w, cy/h, (2*r)/w, (2*r)/h]

# ==================================================
# AUGMENTATION
# ==================================================
def augment(img):
    img = cv2.convertScaleAbs(img, alpha=random.uniform(0.5, 0.8), beta=15)
    img = cv2.GaussianBlur(img, (random.choice([5,7,9]),)*2, 0)

    noise = np.random.normal(0, 4, img.shape).astype(np.float32)
    img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return img

# ==================================================
# DATASET GENERATION (ONE OBJECT PER IMAGE)
# ==================================================
def generate(split, count):
    img_dir = os.path.join(BASE_DIR, f"images/{split}")
    lbl_dir = os.path.join(BASE_DIR, f"labels/{split}")

    for i in range(count):
        img = underwater_bg()
        img = augment(img)

        choice = random.choice(["gate", "mat", "drum", "none"])

        label = None
        if choice == "gate":
            img, label = draw_gate(img)
        elif choice == "mat":
            img, label = draw_mat(img)
        elif choice == "drum":
            img, label = draw_drum(img)

        cv2.imwrite(f"{img_dir}/img_{i}.jpg", img)

        if label:
            with open(f"{lbl_dir}/img_{i}.txt", "w") as f:
                f.write(" ".join(map(str, label)))

    print(f"✅ Generated {count} {split} images")

# ==================================================
# RUN
# ==================================================
if __name__ == "__main__":
    generate("train", 600)
    generate("val", 150)


✅ Generated 600 train images
✅ Generated 150 val images


In [ ]:
!ls /content/drive/MyDrive/AUV_Data/underwater_single_object/images/train | head


img_0.jpg
img_100.jpg
img_101.jpg
img_102.jpg
img_103.jpg
img_104.jpg
img_105.jpg
img_106.jpg
img_107.jpg
img_108.jpg


In [ ]:
!cat /content/drive/MyDrive/AUV_Data/auv_single_object.yaml


path: /content/drive/MyDrive/AUV_Data/underwater_single_object

train: images/train
val: images/val

nc: 3
names:
  0: gate
  1: mat
  2: drum


In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano = fast + enough for PoC


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model.train(
    data="/content/drive/MyDrive/AUV_Data/auv_single_object.yaml",
    epochs=15,
    imgsz=640,
    batch=8,
    device="cpu"
)


Ultralytics 8.3.242 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/AUV_Data/auv_single_object.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspectiv

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/AUV_Data/underwater_single_object/images/val",
    conf=0.5,
    save=True
)


In [ ]:
results = model.predict(
    source="/content/drive/MyDrive/AUV_Data/underwater_single_object/images/val/img_5.jpg",
    conf=0.5
)

r = results[0]

for box in r.boxes:
    cls_id = int(box.cls[0])
    cls_name = model.names[cls_id]

    x1, y1, x2, y2 = box.xyxy[0]
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    print("Object:", cls_name)
    print("Center:", cx.item(), cy.item())
